# BirdCLEF 2026 | 2-Model Ensemble (Personal Models)

effnetv2_b3 v2 ONNX 5-fold + eca_nfnet_l0 SED ONNX 5-fold

**Models:**
1. effnetv2_b3 v2 ONNX 5-fold - 自作モデル (~35min)
2. eca_nfnet_l0 SED ONNX 5-fold - 自作モデル (~35min)

**Target: ~75min total (90min limit)**

In [ ]:
!pip install /kaggle/input/onnxruntime-wheel/onnxruntime-*.whl -q --no-deps 2>/dev/null || pip install /kaggle/input/datasets/ttyn4519/onnxruntime-wheel/onnxruntime-*.whl -q --no-deps

In [ ]:
import os
print("=== /kaggle/input/ contents ===")
if os.path.exists("/kaggle/input"):
    for d in sorted(os.listdir("/kaggle/input")):
        print(f"  {d}/")
        subpath = f"/kaggle/input/{d}"
        if os.path.isdir(subpath):
            for f in sorted(os.listdir(subpath))[:10]:
                print(f"    {f}")
else:
    print("  /kaggle/input does not exist")

In [ ]:
%%writefile effnetv2_b3.py

# ============================================================
# BirdCLEF 2026 | Model 1 Inference
# MODEL: SED EfficientNetV2-B3 v2 ONNX 5-fold (LB 0.886)
# OUTPUT: effnetv2_b3.csv
# ============================================================

import os, re, time, glob, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import onnxruntime as ort
import soundfile as sf
import torch
import torchaudio

warnings.filterwarnings("ignore")

SUBMISSION_FILE = "effnetv2_b3.csv"

SR = 32000
DURATION = 5
N_SAMPLES = SR * DURATION
N_FFT = 2048
N_MELS = 224
F_MIN = 0
F_MAX = 16000
HOP_LENGTH = 512
MEL_NORM = "slaney"
MEL_SCALE = "htk"
NUM_CLASSES = 234

DATA_ROOT = "/kaggle/input/competitions/birdclef-2026"
TEST_DIR = os.path.join(DATA_ROOT, "test_soundscapes")
TRAIN_SC_DIR = os.path.join(DATA_ROOT, "train_soundscapes")
SAMPLE_SUBMISSION_CSV = os.path.join(DATA_ROOT, "sample_submission.csv")

# Find model directory (path varies between direct dataset and nested datasets)
MODEL_DIR_CANDIDATES = [
    "/kaggle/input/sed-effnetv2-b3-models",
    "/kaggle/input/datasets/ttyn4519/sed-effnetv2-b3-models",
]
MODEL_DIR = None
for d in MODEL_DIR_CANDIDATES:
    if os.path.isdir(d):
        MODEL_DIR = d
        break
assert MODEL_DIR is not None, f"Model dir not found in: {MODEL_DIR_CANDIDATES}"
print(f"Model dir: {MODEL_DIR}")

ONNX_PATHS = [f"{MODEL_DIR}/sed_effnetv2_b3_fold{fold}_best.onnx" for fold in range(5)]

def create_ort_session(onnx_path):
    sess_options = ort.SessionOptions()
    sess_options.intra_op_num_threads = 4
    sess_options.inter_op_num_threads = 1
    return ort.InferenceSession(onnx_path, sess_options, providers=["CPUExecutionProvider"])

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, f_min=F_MIN, f_max=F_MAX, power=2.0,
    norm=MEL_NORM, mel_scale=MEL_SCALE,
)
amp_to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)

@torch.no_grad()
def compute_mel(waveforms):
    mel = mel_transform(waveforms)
    mel = amp_to_db(mel)
    B = mel.shape[0]
    mel_flat = mel.reshape(B, -1)
    mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
    mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
    mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
    return mel.unsqueeze(1).repeat(1, 3, 1, 1).numpy()

def main():
    t_total_start = time.time()

    sub_df = pd.read_csv(SAMPLE_SUBMISSION_CSV)
    target_columns = [c for c in sub_df.columns if c != "row_id"]
    print(f"Target columns: {len(target_columns)} classes")

    row_ids = sub_df["row_id"].values
    end_seconds = np.array([int(rid.rsplit("_", 1)[1]) for rid in row_ids])
    filenames = np.array([rid.rsplit("_", 1)[0] + ".ogg" for rid in row_ids])

    unique_files = list(dict.fromkeys(filenames))
    print(f"Unique soundscape files: {len(unique_files)}")

    existing_files = [f for f in unique_files if os.path.isfile(f"{TEST_DIR}/{f}")]
    if not existing_files:
        existing_files = [f for f in unique_files if os.path.isfile(f"{TRAIN_SC_DIR}/{f}")]
        soundscape_dir = TRAIN_SC_DIR
        print(f"No test .ogg found; using {len(existing_files)} train_soundscapes fallback files.")
    else:
        soundscape_dir = TEST_DIR
        if len(existing_files) < len(unique_files):
            print(f"  Available test files: {len(existing_files)}/{len(unique_files)}")
    unique_files = existing_files

    # Load ONNX models (skip if no files to process - Save & Run mode)
    sessions = []
    if len(unique_files) > 0:
        for fold_idx, onnx_path in enumerate(ONNX_PATHS):
            print(f"Loading ONNX model fold {fold_idx}: {onnx_path}")
            sessions.append(create_ort_session(onnx_path))
        input_name = sessions[0].get_inputs()[0].name
        print(f"  ONNX input name: {input_name}")
    else:
        print("No files to process, skipping model loading.")

    row_id_to_idx = {rid: i for i, rid in enumerate(row_ids)}
    predictions = sub_df[target_columns].values.astype(np.float32).copy()

    n_processed_files = 0
    n_processed_segments = 0
    t_inference_start = time.time()

    for file_idx, fname in enumerate(unique_files):
        filepath = f"{soundscape_dir}/{fname}"
        file_mask = filenames == fname
        file_end_seconds = end_seconds[file_mask]
        file_row_ids = row_ids[file_mask]

        audio, file_sr = sf.read(filepath, dtype="float32")
        if audio.ndim == 2:
            audio = audio.mean(axis=1)

        segments = []
        for end_sec in file_end_seconds:
            start_sample = (end_sec - DURATION) * SR
            end_sample = end_sec * SR
            if start_sample < 0:
                seg = np.zeros(N_SAMPLES, dtype=np.float32)
                available = audio[:max(end_sample, 0)]
                if len(available) > 0: seg[N_SAMPLES - len(available):] = available
            elif end_sample > len(audio):
                seg = np.zeros(N_SAMPLES, dtype=np.float32)
                available = audio[min(start_sample, len(audio)):]
                if len(available) > 0: seg[:len(available)] = available
            else:
                seg = audio[start_sample:end_sample]
            segments.append(seg)

        mel_batch = compute_mel(torch.from_numpy(np.stack(segments)).float())

        clipwise_probs_sum = np.zeros((len(segments), NUM_CLASSES), dtype=np.float32)
        for session in sessions:
            for i in range(mel_batch.shape[0]):
                outputs = session.run(None, {input_name: mel_batch[i:i+1]})
                clipwise_probs_sum[i] += outputs[0][0]
        clipwise_probs = clipwise_probs_sum / len(sessions)

        for i, rid in enumerate(file_row_ids):
            predictions[row_id_to_idx[rid]] = clipwise_probs[i]

        n_processed_files += 1
        n_processed_segments += len(file_end_seconds)
        if n_processed_files % 100 == 0 or n_processed_files == len(unique_files):
            elapsed = time.time() - t_inference_start
            speed = n_processed_segments / elapsed if elapsed > 0 else 0
            print(f"  [{n_processed_files}/{len(unique_files)}] {n_processed_segments} segments, {elapsed:.1f}s, {speed:.1f} seg/s")

    sub_df[target_columns] = predictions
    sub_df.set_index("row_id").to_csv(SUBMISSION_FILE)

    t_total = time.time() - t_total_start
    print(f"\nInference complete: {n_processed_files} files, {n_processed_segments} segments, {t_total:.1f}s ({t_total/60:.1f}min)")
    print(f"Output: {SUBMISSION_FILE}")

if __name__ == "__main__":
    main()

In [ ]:
%%writefile eca_nfnet_l0.py

# ============================================================
# BirdCLEF 2026 | Model 2 Inference
# MODEL: SED eca_nfnet_l0 ONNX 5-fold (LB 0.885)
# OUTPUT: eca_nfnet_l0.csv
# ============================================================

import os, re, time, glob, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import onnxruntime as ort
import soundfile as sf
import torch
import torchaudio

warnings.filterwarnings("ignore")

SUBMISSION_FILE = "eca_nfnet_l0.csv"

SR = 32000
DURATION = 5
N_SAMPLES = SR * DURATION
N_FFT = 2048
N_MELS = 224
F_MIN = 0
F_MAX = 16000
HOP_LENGTH = 512
MEL_NORM = "slaney"
MEL_SCALE = "htk"
NUM_CLASSES = 234

DATA_ROOT = "/kaggle/input/competitions/birdclef-2026"
TEST_DIR = os.path.join(DATA_ROOT, "test_soundscapes")
TRAIN_SC_DIR = os.path.join(DATA_ROOT, "train_soundscapes")
SAMPLE_SUBMISSION_CSV = os.path.join(DATA_ROOT, "sample_submission.csv")

# Find model directory (path varies between direct dataset and nested datasets)
MODEL_DIR_CANDIDATES = [
    "/kaggle/input/sed-eca-nfnet-l0-models",
    "/kaggle/input/datasets/ttyn4519/sed-eca-nfnet-l0-models",
]
MODEL_DIR = None
for d in MODEL_DIR_CANDIDATES:
    if os.path.isdir(d):
        MODEL_DIR = d
        break
assert MODEL_DIR is not None, f"Model dir not found in: {MODEL_DIR_CANDIDATES}"
print(f"Model dir: {MODEL_DIR}")

ONNX_PATHS = [f"{MODEL_DIR}/sed_eca_nfnet_l0_fold{fold}_best.onnx" for fold in range(5)]

def create_ort_session(onnx_path):
    sess_options = ort.SessionOptions()
    sess_options.intra_op_num_threads = 4
    sess_options.inter_op_num_threads = 1
    return ort.InferenceSession(onnx_path, sess_options, providers=["CPUExecutionProvider"])

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, f_min=F_MIN, f_max=F_MAX, power=2.0,
    norm=MEL_NORM, mel_scale=MEL_SCALE,
)
amp_to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)

@torch.no_grad()
def compute_mel(waveforms):
    mel = mel_transform(waveforms)
    mel = amp_to_db(mel)
    B = mel.shape[0]
    mel_flat = mel.reshape(B, -1)
    mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
    mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
    mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
    return mel.unsqueeze(1).repeat(1, 3, 1, 1).numpy()

def main():
    t_total_start = time.time()

    sub_df = pd.read_csv(SAMPLE_SUBMISSION_CSV)
    target_columns = [c for c in sub_df.columns if c != "row_id"]
    print(f"Target columns: {len(target_columns)} classes")

    row_ids = sub_df["row_id"].values
    end_seconds = np.array([int(rid.rsplit("_", 1)[1]) for rid in row_ids])
    filenames = np.array([rid.rsplit("_", 1)[0] + ".ogg" for rid in row_ids])

    unique_files = list(dict.fromkeys(filenames))
    print(f"Unique soundscape files: {len(unique_files)}")

    existing_files = [f for f in unique_files if os.path.isfile(f"{TEST_DIR}/{f}")]
    if not existing_files:
        existing_files = [f for f in unique_files if os.path.isfile(f"{TRAIN_SC_DIR}/{f}")]
        soundscape_dir = TRAIN_SC_DIR
        print(f"No test .ogg found; using {len(existing_files)} train_soundscapes fallback files.")
    else:
        soundscape_dir = TEST_DIR
        if len(existing_files) < len(unique_files):
            print(f"  Available test files: {len(existing_files)}/{len(unique_files)}")
    unique_files = existing_files

    # Load ONNX models (skip if no files to process - Save & Run mode)
    sessions = []
    if len(unique_files) > 0:
        for fold_idx, onnx_path in enumerate(ONNX_PATHS):
            print(f"Loading ONNX model fold {fold_idx}: {onnx_path}")
            sessions.append(create_ort_session(onnx_path))
        input_name = sessions[0].get_inputs()[0].name
        print(f"  ONNX input name: {input_name}")
    else:
        print("No files to process, skipping model loading.")

    row_id_to_idx = {rid: i for i, rid in enumerate(row_ids)}
    predictions = sub_df[target_columns].values.astype(np.float32).copy()

    n_processed_files = 0
    n_processed_segments = 0
    t_inference_start = time.time()

    for file_idx, fname in enumerate(unique_files):
        filepath = f"{soundscape_dir}/{fname}"
        file_mask = filenames == fname
        file_end_seconds = end_seconds[file_mask]
        file_row_ids = row_ids[file_mask]

        audio, file_sr = sf.read(filepath, dtype="float32")
        if audio.ndim == 2:
            audio = audio.mean(axis=1)

        segments = []
        for end_sec in file_end_seconds:
            start_sample = (end_sec - DURATION) * SR
            end_sample = end_sec * SR
            if start_sample < 0:
                seg = np.zeros(N_SAMPLES, dtype=np.float32)
                available = audio[:max(end_sample, 0)]
                if len(available) > 0: seg[N_SAMPLES - len(available):] = available
            elif end_sample > len(audio):
                seg = np.zeros(N_SAMPLES, dtype=np.float32)
                available = audio[min(start_sample, len(audio)):]
                if len(available) > 0: seg[:len(available)] = available
            else:
                seg = audio[start_sample:end_sample]
            segments.append(seg)

        mel_batch = compute_mel(torch.from_numpy(np.stack(segments)).float())

        clipwise_probs_sum = np.zeros((len(segments), NUM_CLASSES), dtype=np.float32)
        for session in sessions:
            for i in range(mel_batch.shape[0]):
                outputs = session.run(None, {input_name: mel_batch[i:i+1]})
                clipwise_probs_sum[i] += outputs[0][0]
        clipwise_probs = clipwise_probs_sum / len(sessions)

        for i, rid in enumerate(file_row_ids):
            predictions[row_id_to_idx[rid]] = clipwise_probs[i]

        n_processed_files += 1
        n_processed_segments += len(file_end_seconds)
        if n_processed_files % 100 == 0 or n_processed_files == len(unique_files):
            elapsed = time.time() - t_inference_start
            speed = n_processed_segments / elapsed if elapsed > 0 else 0
            print(f"  [{n_processed_files}/{len(unique_files)}] {n_processed_segments} segments, {elapsed:.1f}s, {speed:.1f} seg/s")

    sub_df[target_columns] = predictions
    sub_df.set_index("row_id").to_csv(SUBMISSION_FILE)

    t_total = time.time() - t_total_start
    print(f"\nInference complete: {n_processed_files} files, {n_processed_segments} segments, {t_total:.1f}s ({t_total/60:.1f}min)")
    print(f"Output: {SUBMISSION_FILE}")

if __name__ == "__main__":
    main()

## 2-Model Ensemble Blend

In [ ]:
%%writefile blend.py

# ============================================================
# BirdCLEF 2026 | 2-Model Ensemble Blend
# Reads effnetv2_b3.csv, eca_nfnet_l0.csv -> submission.csv
# ============================================================

import os
import re
import numpy as np
import pandas as pd

DATA_ROOT  = "/kaggle/input/competitions/birdclef-2026"
sample_sub = pd.read_csv(os.path.join(DATA_ROOT, "sample_submission.csv"))
SPECIES    = list(sample_sub.columns[1:])
expected_ids = sample_sub["row_id"].tolist()

# Load 2 model predictions
df_b3    = pd.read_csv("effnetv2_b3.csv", index_col="row_id")[SPECIES]
df_nfnet = pd.read_csv("eca_nfnet_l0.csv", index_col="row_id")[SPECIES]

preds_b3    = df_b3.values
preds_nfnet = df_nfnet.values

# Weighted blend: LB-proportional weights
# 0.886 + 0.885 = 1.771
W_B3    = 0.886 / 1.771
W_NFNET = 0.885 / 1.771

print(f"Blend weights: effnetv2_b3={W_B3:.3f}, eca_nfnet_l0={W_NFNET:.3f}")

blended = W_B3 * preds_b3 + W_NFNET * preds_nfnet

# Post-processing on blended result
FILE_MAX_WEIGHT = 0.05
SHARPEN_POWER = 1.5

# Group by file for post-processing
row_pattern = re.compile(r"^(.*)_(\d+)$")
row_ids = df_b3.index.tolist()
stems = []
for rid in row_ids:
    m = row_pattern.match(str(rid))
    stems.append(m.group(1) if m else rid)

unique_stems = list(dict.fromkeys(stems))
stem_to_indices = {}
for i, s in enumerate(stems):
    stem_to_indices.setdefault(s, []).append(i)

for stem in unique_stems:
    indices = stem_to_indices[stem]
    n = len(indices)
    preds = blended[indices]

    # Confidence-Sharpened Smoothing
    if n > 4:
        weights = np.array([0.05, 0.15, 0.60, 0.15, 0.05])
        pad = 2
        preds_sharp = np.power(preds, SHARPEN_POWER)
        padded = np.pad(preds_sharp, ((pad, pad), (0, 0)), mode='edge')
        smoothed = np.zeros_like(preds_sharp)
        for i_w in range(n):
            for j, w in enumerate(weights):
                smoothed[i_w] += w * padded[i_w + j]
        preds = np.power(smoothed, 1.0 / SHARPEN_POWER)
    elif n > 2:
        weights = np.array([0.20, 0.60, 0.20])
        pad = 1
        padded = np.pad(preds, ((pad, pad), (0, 0)), mode='edge')
        smoothed = np.zeros_like(preds)
        for i_w in range(n):
            for j, w in enumerate(weights):
                smoothed[i_w] += w * padded[i_w + j]
        preds = smoothed

    # File-Max Prior
    file_max = np.max(preds, axis=0, keepdims=True)
    preds = preds + FILE_MAX_WEIGHT * file_max

    blended[indices] = preds

# Build submission
submission = pd.DataFrame(blended, columns=SPECIES, index=df_b3.index)
submission = submission.loc[expected_ids].reset_index()
submission.to_csv("submission.csv", index=False)
print(f"Saved: submission.csv | {submission.shape}")

## Run All

In [ ]:
import time
t_start = time.time()

print("=" * 60)
print("MODEL 1: effnetv2_b3 v2 ONNX 5-fold (LB 0.886)")
print("=" * 60)
!python effnetv2_b3.py

print("=" * 60)
print("MODEL 2: eca_nfnet_l0 ONNX 5-fold (LB 0.885)")
print("=" * 60)
!python eca_nfnet_l0.py

print("=" * 60)
print("BLEND")
print("=" * 60)
!python blend.py

t_total = time.time() - t_start
print(f"\nTotal pipeline time: {t_total:.1f}s ({t_total/60:.1f}min)")

In [ ]:
import os
assert os.path.isfile("submission.csv"), "ERROR: submission.csv was not created!"
import pandas as pd
df = pd.read_csv("submission.csv")
print(f"submission.csv: {df.shape}")
print(f"First 3 rows:\n{df.iloc[:3, :6].to_string()}")